In [1]:
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
import torchvision.transforms as transforms
from torch.utils.data import random_split
from src.utils.custom_dataset import *

In [2]:
PROCESSED_PATH = 'dataset/processed/'
IN = "IN/"
OUT = "OUT/"
        
ux_inputs = np.load(PROCESSED_PATH+IN+"ux.npy")[:,:,:,np.newaxis]
ux_inputs.shape

(1810, 51, 51, 1)

In [3]:
# class SelfAttention(nn.Module):
#     def __init__(self, input_dim, hidden_dim):
#         super(SelfAttention, self).__init__()
#         self.hidden_dim = hidden_dim
        
#         self.query = nn.Linear(input_dim, hidden_dim)
#         self.key = nn.Linear(input_dim, hidden_dim)
#         self.value = nn.Linear(input_dim, hidden_dim)
#         self.softmax = nn.Softmax(dim=2)
#         self.batch_norm = nn.BatchNorm1d(hidden_dim)
        
#     def forward(self, x):
        
#         q = self.query(x.float())
#         k = self.key(x.float())
#         v = self.value(x.float())
        
#         scores = torch.matmul(q, k.transpose(1, 2))
#         # print("scores shape : ",scores.shape)
#         attention_weights = self.softmax(scores)
#         # print("attention_weights shape : ",attention_weights.shape)
#         attended_values = torch.matmul(attention_weights, v)
#         # print("attended_values shape ",attended_values.shape)
#         # attended_values = self.batch_norm(attended_values)
#         # print("attended_values shape ",attended_values.shape)
        
#         return attended_values

# class Model(nn.Module):
#     def __init__(self, input_dim, hidden_dim, output_dim):
#         super(Model, self).__init__()
        
#         self.self_attention = SelfAttention(input_dim, hidden_dim)
#         self.fc1 = nn.Linear(hidden_dim, 128) 
#         self.fc2 = nn.Linear(128, 64) 
#         self.fc3 = nn.Linear(64, output_dim)  
        
#     def forward(self, x):
#         attended_values = self.self_attention(x)
#         temp = self.fc1(attended_values.squeeze())
#         temp = self.fc2(temp)
#         output = self.fc3(temp)
        
#         return output

# class CustomDataset(Dataset):
#     def __init__(self, input_data, output_data):
#         self.input_data = input_data
#         self.output_data = output_data
        
#     def __getitem__(self, index):
#         inputs = self.input_data[index]
#         outputs = self.output_data[index]
#         item = {'input':inputs,'output':outputs}
#         return item
    
#     def __len__(self):
#         return len(self.input_data)

### 1. 데이터세트 지정

- dataset 지정 (Customdataset Class 사용)

In [4]:
dataset = CustomDataset(transform=transforms.Compose([transforms.ToTensor()]),
                        target_transform=transforms.Compose([transforms.ToTensor()]))

- dataset 분할 (random_split 사용)

In [5]:
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
validation_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - validation_size

train_dataset, validation_dataset, test_dataset = random_split(dataset,[train_size,validation_size,test_size],
                                                               generator=torch.Generator().manual_seed(42))
print(f"Training Data Size : {len(train_dataset)}")
print(f"Validation Data Size : {len(validation_dataset)}")
print(f"Testing Data Size : {len(test_dataset)}")

Training Data Size : 1448
Validation Data Size : 181
Testing Data Size : 181


- DataLoader 설정

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=True)

### 2. 모델 로드

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from src.model.model import *
model = TransUNet().to(device)

In [10]:
def train_loop(dataloader,model,loss_fn,optimizer):
    
    size = len(dataloader.dataset) # 입력 데이터세트의 총 길이  : 320
    num_batches = len(dataloader) # 배치 총 개수 (320/4 = 80) 
    train_loss = 0 # loss 초기화

    # train_loop 
    # ux_input,uy_input,p_input,bdr_input,st_input,sv_input,\
    # ux_output,uy_output,p_output

    for batch, (ux_in,uy_in,p_in,bdr,st,sv,ux_out,uy_out,p_out) in enumerate(tqdm(dataloader)): 
        ux_in = ux_in.type(torch.float32).to(device)    # torch.Size([4, 1, 51, 51])
        uy_in = uy_in.type(torch.float32).to(device)    # torch.Size([4, 1, 51, 51])
        p_in = p_in.type(torch.float32).to(device)      # torch.Size([4, 1, 51, 51])
        bdr = bdr.type(torch.float32).to(device)        # torch.Size([4, 1, 51, 51])
        st = st.type(torch.int32).to(device)            # torch.Size([4, 4, 1]), embedding 으로 넘어가기때문에 int 형
        sv = sv.type(torch.float32).to(device)          # torch.Size([4, 4, 1])
        ux_out = ux_out.type(torch.float32).to(device)  # torch.Size([4, 1, 51, 51])
        uy_out = uy_out.type(torch.float32).to(device)  # torch.Size([4, 1, 51, 51])
        p_out = p_out.type(torch.float32).to(device)    # torch.Size([4, 1, 51, 51])

        inp = torch.concat([ux_in,uy_in,p_in,bdr],axis=1)
        X = [inp,st,sv]
        y = torch.concat([ux_out,uy_out,p_out],axis=1)
        
        # 모델 학습
        pred = model(X)
        loss = loss_fn(pred,y)
        optimizer.zero_grad() # 이전 루프에서 각 파라미터들에 저장된 .grad를 초기화
        loss.backward() # 각 파라미터들의 .grad값에 변화정도가 저장이 됨.
        optimizer.step() # 최적화함수에 맞게, 각 파라미터 업데이트

        # Loss 계산
        train_loss += loss.item() # item()을 통해, 변수에서 값만 가져옴

    train_loss /= num_batches
    print(f"Train loss: {train_loss:>8f} \n")
    
    return train_loss

def validation_loop(dataloader,model,loss_fn):

    size = len(dataloader.dataset) # 입력 데이터세트의 총 길이 : 40
    num_batches = len(dataloader) # 배치 총 개수 (40/4 = 10)
    validation_loss = 0 # loss 초기화

    with torch.no_grad():
        for ux_in,uy_in,p_in,bdr,st,sv,ux_out,uy_out,p_out in tqdm(dataloader):
            ux_in = ux_in.type(torch.float32).to(device)
            uy_in = uy_in.type(torch.float32).to(device)
            p_in = p_in.type(torch.float32).to(device)
            bdr = bdr.type(torch.float32).to(device)
            st = st.type(torch.int32).to(device) # embedding 으로 넘어가기때문에 int 형
            sv = sv.type(torch.float32).to(device)
            ux_out = ux_out.type(torch.float32).to(device)
            uy_out = uy_out.type(torch.float32).to(device)
            p_out = p_out.type(torch.float32).to(device)


            inp = torch.concat([ux_in,uy_in,p_in,bdr],axis=1)
            X = [inp,st,sv]
            y = torch.concat([ux_out,uy_out,p_out],axis=1)
            
            # 모델 예측 & Loss 계산
            pred = model(X)
            loss = loss_fn(pred,y)
            validation_loss += loss.item() # item()을 통해, 변수에서 값만 가져옴
            
    validation_loss /= num_batches
    print(f"Validation loss: {validation_loss:>8f} \n")
    return validation_loss

In [11]:
learning_rate = 1e-3
epochs = 200
loss_fn = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

train_loss_all = []
validation_loss_all = []

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer)
    train_loss_all.append(train_loss)
    validation_loss = validation_loop(validation_dataloader, model, loss_fn)
    validation_loss_all.append(validation_loss)
print("학습완료")

Epoch 1
-------------------------------


100%|██████████| 362/362 [00:06<00:00, 58.56it/s]


Train loss: 0.029246 



100%|██████████| 46/46 [00:00<00:00, 348.45it/s]


Validation loss: 0.022808 

Epoch 2
-------------------------------


...

Validation loss: 0.001927 

Epoch 200
-------------------------------


100%|██████████| 362/362 [00:06<00:00, 57.51it/s]


Train loss: 0.002249 



100%|██████████| 46/46 [00:00<00:00, 299.66it/s]

Validation loss: 0.002191 

학습완료
